# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20, [10,20,30] , s["b"] это второй элемент, iloc[0] показывает 1ую строку, все элементы в строке, значит все элементы
2. Алматы: NaN
   Шымкент: 3
   Астана: NaN
   Мы не пропускаем элементы, и если элемент отсутствует в 1 списке, сумма будет NaN
3. (2, 2), элемент удалился, кол-во столбцов полных стало 2, строк также 2
4. Series, DataFrame (1 скобки series, 2 скобки DataFrame)
5. 3, 1 удалился, значит осталось [2,3] , 1 элемент это 3 (с 0 если считать)

In [1]:
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])

20 10
Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64
(2, 2)
Series DataFrame
[1, 2] 5


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [4]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [8]:
# 1
df.info()
#Пропусков в delivery days : 5463 - 5303 = 160.
print(df["city"].nunique())
print(df["category"].nunique())
print(df["status"].nunique())
print(df["customer_id"].nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   object 
 2   customer_id    5463 non-null   object 
 3   city           5463 non-null   object 
 4   category       5463 non-null   object 
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 341.6+ KB
14
5
2
2003


Сообщение менеджеру:
Есть пропуски в delivery_days, поэтому статистика по доставкам может быть некорректна, также есть 14 значений городов, хотя городов всего лишь 6, поэтому один город написан несколькими вариантами. Клиентов 2003, хотя заказов 5463, поэтому могут быть повторения, и надо проверить тестовых клиентов

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [19]:
# 2
df2 = df[df["amount"] != "TEST"]["amount"]
print(df2.describe())
df2.value_counts().head(5)

count    5.463000e+03
mean     2.011411e+04
std      2.343251e+05
min      7.200000e+02
25%      5.870000e+03
50%      1.032000e+04
75%      1.869500e+04
max      9.999990e+06
Name: amount, dtype: float64


,count
amount,
5410.0,11
5460.0,10
8550.0,10
4490.0,9
2690.0,9


Честнее показать медиану, чем среднее в большинстве вариантов, особенно если мы говорим про средний чек. Потому что всегда есть outliers которые делают среднее слишком большим, а медиана нам показывает что 50% чеков меньше 10320, что позволяет нам делать какие то выводы.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [26]:
# 3
SH = ["Шымкент", "шымкент", "Shymkent"]
print(df["city"].unique())
sh = df[df["city"].isin(SH)]
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
al = df[df["city"].isin(AL)]

cancelled = df[df["status"] == "отменён"]
print("Доля по всем заказам:", len(cancelled) / len(df) * 100 , "%")
cancelled_sh = sh[sh["status"] == "отменён"]
cancelled_al = al[al["status"] == "отменён"]
print("Доля по Шымкенту:", len(cancelled_sh) / len(sh) * 100, "%")
print("Доля по Алматы:", len(cancelled_al) / len(al) * 100, "%")

['Алматы' 'Астана' 'Караганда' 'Шымкент' 'алматы' 'Атырау' 'шымкент'
 'Shymkent' 'астана' 'Актобе' 'Astana ' 'АЛМАТЫ' ' Алматы' 'Almaty']
Доля по всем заказам: 10.232473000183049 %
Доля по Шымкенту: 20.398009950248756 %
Доля по Алматы: 7.861060329067642 %


Выделяется Шымкент, 20% заказов отменено, учитывая что по всем заказам 10%, процент почти в 2 раза больше.

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [43]:
# 4
paid_sep_sh = df[df["city"].isin(SH) & paid & sep & not_test]["amount"]
paid_octo_sh = df[df["city"].isin(SH) & paid & octo & not_test]["amount"]
print((paid_octo_sh.sum() / paid_sep_sh.sum() - 1) * 100, "%")

paid_sep_al = df[df["city"].isin(AL) & paid & sep & not_test]["amount"]
paid_octo_al = df[df["city"].isin(AL) & paid & octo & not_test]["amount"]
print((paid_octo_al.sum() / paid_sep_al.sum() - 1) * 100, "%")

-18.29952725362641 %
-5.674590342400188 %


В шымкенте сильнее падение, -18%, почти в три раза больше чем в Алмате, надо посмотреть почему в Шымкенте упала выручка, что можно с этим сделать


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [50]:
# 5
cancel_sh = df[df["city"].isin(SH) & (df["status"] == "отменён")]
print(len(cancel_sh))


164


Во первых забыли учесть все варианты Шымкента через SH, и нужно было обернуть в скобки каждое условие чтобы код заработал, потому что он считал как "Шымкент" & df["status"], а не отдельно как нам нужно

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [54]:
# 6
print(df["customer_id"].value_counts().head(1))
print(df.loc[df["customer_id"] == "C01818",["order_id","order_date", "city", "amount", "status"]])


customer_id
C01818    11
Name: count, dtype: int64
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен


Есть, заказ 101504, это значит что некоторая выручка считалась дважды, что увеличивало нашу выручку

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [74]:
# 7
paid_not_test = df[paid & not_test]
Q3 = np.percentile(paid_not_test["amount"], 75)
Q1 = np.percentile(paid_not_test["amount"], 25)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
df_upper = df[df["amount"] > upper_bound]
print(len(df_upper))
print("Доля: ", len(df_upper) / len(df) * 100, "%")
print(df_upper.value_counts("category"))

351
Доля:  6.425041186161449 %
category
Электроника    338
Одежда          11
Дом              2
Name: count, dtype: int64


## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [75]:
# 8
print(cancelled.value_counts("city"))
print(df.value_counts("city"))

city
Алматы       154
Шымкент      153
Астана       104
Караганда     48
Актобе        41
Атырау        11
астана        11
Shymkent       9
алматы         9
Astana         8
Almaty         4
 Алматы        3
АЛМАТЫ         2
шымкент        2
Name: count, dtype: int64
city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64


Нет, нельзя судить только по одному лишь названию, надо группировать города как мы делали до этого, а не разделять их по разным ярлыкам.